In [1]:
from typing import TypedDict, Literal

class Portfolio(TypedDict):
    amount_usd: float
    total_usd: float
    target_currency: Literal["INR", "EUR"]
    total: float


In [2]:
def calc_total(state: Portfolio) -> Portfolio:
    state['total_usd'] = state['amount_usd'] * 1.08
    return state

def convert_to_inr(state: Portfolio) -> Portfolio:
    state['total'] = state['total_usd'] * 94.71
    return state

def convert_to_eur(state: Portfolio) -> Portfolio:
    state['total'] = state['total_usd'] * 0.87
    return state


In [3]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(Portfolio)

In [4]:
builder.add_node("calc_total_node", calc_total)
builder.add_node("convert_to_inr_node", convert_to_inr)
builder.add_node("convert_to_eur_node", convert_to_eur)

def choose_conversion(state: Portfolio) -> str:
    return state['target_currency']

builder.add_edge(START, "calc_total_node")
builder.add_conditional_edges("calc_total_node", choose_conversion, {
    "INR": "convert_to_inr_node",
    "EUR": "convert_to_eur_node"
})
builder.add_edge(["convert_to_eur_node", "convert_to_inr_node"], END)

graph = builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))